## Prepare Env

In [ ]:
%pip install boto3 pyspark delta-spark python-dotenv

In [ ]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Define S3 storage
obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'demo-access-key')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'demo-secret-key')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'http://localhost:9000')

## Ingestion
>### 1. Copy files from local to data lake layer files

In [ ]:
path_to_files = "/home/huy/Documents/ICTTM/test-code/data/Vietnam-20231201T205237Z-001/Vietnam/Import/Vietnam Import 2019.csv"
bucket_name = "warehouse"
folder_target = "files/bol.file"

In [ ]:
# Create an S3 client with MinIO configuration
s3_client = boto3.client(
    's3',
    aws_access_key_id=obj_storage_access_key,
    aws_secret_access_key=obj_storage_secret_key,
    endpoint_url=obj_storage_endpoint
)

In [ ]:
# Upload the file to the MinIO bucket
filename = path_to_files.split("/")[-1]
with open(path_to_files, 'rb') as data:
    s3_client.upload_fileobj(data, bucket_name, f"{folder_target}/{filename}")

print(f"File uploaded to MinIO bucket: {bucket_name}")

> ### 2. Layer files to layer bronze
Write files which are in layer files to delta table in layer bronze

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("JsonToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [ ]:
filename = path_to_files.split("/")[-1]
file_path = f"s3a://{bucket_name}/{folder_target}/{filename}"
table_name = f"bronze/{filename.split('.')[0]}.delta"

delta_table_path = f"s3a://{bucket_name}/{table_name}"

In [ ]:
# Read file into a DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

In [ ]:
# Lowercase and replace spaces with underscores for all column names
new_columns = [col(old_col).alias(old_col.lower().replace(' ', '_').replace('(', '|').replace(')', '|')) for old_col in df.columns]
df = df.select(*new_columns)

In [ ]:
# df.show()
df.columns

In [ ]:
# Write DataFrame to Delta table
df.write.format("delta").mode("overwrite").save(delta_table_path)

# Stop the Spark session
spark.stop()

# Processing

Processing these step before writing data to layer silver
1. Transform to standard schema of layer silver
2. Unique each record
3. Add fields
4. Map entities
5. Upsert